## Import dan Direktori Project

In [1]:
from pathlib import Path
import pandas as pd
import json
import sys
import importlib
import subprocess

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

direktori_aktif = Path.cwd().resolve()

if direktori_aktif.name == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_src = direktori_project / "src"
direktori_models = direktori_project / "models"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_intelligence = direktori_project / "data" / "intelligence"
direktori_samples_metadata = direktori_project / "data" / "samples_metadata"
direktori_examples = direktori_project / "examples"

for folder in [
    direktori_src,
    direktori_outputs,
    direktori_examples
]:
    folder.mkdir(parents=True, exist_ok=True)

if str(direktori_src) not in sys.path:
    sys.path.append(str(direktori_src))

print("Direktori aktif notebook:", direktori_aktif)
print("Direktori project:", direktori_project)
print("Folder src:", direktori_src)
print("Folder outputs:", direktori_outputs)
print("Folder examples:", direktori_examples)

Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Direktori project: C:\Users\ASUS\PHISHING
Folder src: C:\Users\ASUS\PHISHING\src
Folder outputs: C:\Users\ASUS\PHISHING\reports\outputs
Folder examples: C:\Users\ASUS\PHISHING\examples


## Validasi File Penting

In [2]:
daftar_file_wajib_step10 = [
    direktori_src / "phishrisk_engine_v3.py",
    direktori_src / "url_intelligence.py",
    direktori_src / "file_static_analyzer.py",
    direktori_models / "model_terbaik_intelligence_v2.pkl",
    direktori_outputs / "daftar_fitur_intelligence_v2.json",
    direktori_intelligence / "official_domains_global.csv",
    direktori_intelligence / "brand_keywords_global.csv",
    direktori_intelligence / "suspicious_keywords_global.csv",
    direktori_intelligence / "suspicious_file_rules.csv"
]

hasil_validasi_awal_step10 = []

for lokasi_file in daftar_file_wajib_step10:
    hasil_validasi_awal_step10.append({
        "nama_file": lokasi_file.name,
        "lokasi": str(lokasi_file),
        "tersedia": lokasi_file.exists(),
        "ukuran_kb": round(lokasi_file.stat().st_size / 1024, 2) if lokasi_file.exists() else 0
    })

data_validasi_awal_step10 = pd.DataFrame(hasil_validasi_awal_step10)

if not data_validasi_awal_step10["tersedia"].all():
    display(data_validasi_awal_step10)
    raise FileNotFoundError("Ada file wajib STEP 10 yang belum tersedia.")

print("Semua file wajib STEP 10 tersedia.")
data_validasi_awal_step10

Semua file wajib STEP 10 tersedia.


,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v3.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py,True,17.66
1,url_intelligence.py,C:\Users\ASUS\PHISHING\src\url_intelligence.py,True,15.60
2,file_static_analyzer.py,C:\Users\ASUS\PHISHING\src\file_static_analyze...,True,16.67
3,model_terbaik_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_terbaik_in...,True,49114.43
4,daftar_fitur_intelligence_v2.json,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,1.02
5,official_domains_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\offic...,True,4.31
6,brand_keywords_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\brand...,True,1.43
7,suspicious_keywords_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\suspi...,True,1.78
8,suspicious_file_rules.csv,C:\Users\ASUS\PHISHING\data\intelligence\suspi...,True,1.74


## Perbaikan Pembersihan URL dari File

In [3]:
kode_pembersih_url_file = r'''

# URL Cleaner V3.1
# Dokumentasi: membersihkan URL hasil ekstraksi dari teks/binary.

def bersihkan_url_terekstrak(url):
    url = str(url).strip()
    url = url.replace("\\x00", "")
    url = url.replace("\x00", "")

    pemotong = [
        "PK\x03\x04",
        "PK\\x03\\x04",
        "%%EOF",
        "<",
        ">",
        "\"",
        "'",
        "\\",
        "{",
        "}",
        "[",
        "]"
    ]

    for token in pemotong:
        if token in url:
            url = url.split(token)[0]

    pola_valid = r"^(https?://[a-zA-Z0-9\-._~:/?#\[\]@!$&()*+,;=%]+)"
    cocok = re.match(pola_valid, url)

    if cocok:
        url = cocok.group(1)

    url = url.rstrip(".,;:)")
    return url


def ekstrak_url_dari_teks(teks):
    teks = str(teks)
    pola_url = r"https?://[^\s<>'\"\\)\]\}]+"
    daftar_url = re.findall(pola_url, teks, flags=re.IGNORECASE)

    daftar_bersih = []

    for url in daftar_url:
        url_bersih = bersihkan_url_terekstrak(url)

        if not url_bersih:
            continue

        if "PK" in url_bersih[-6:]:
            url_bersih = url_bersih.split("PK")[0].rstrip(".,;:")

        if url_bersih not in daftar_bersih:
            daftar_bersih.append(url_bersih)

    return daftar_bersih
'''

lokasi_file_static_analyzer = direktori_src / "file_static_analyzer.py"

with open(lokasi_file_static_analyzer, "a", encoding="utf-8") as file:
    file.write(kode_pembersih_url_file)

print("Pembersih URL File Static Analyzer berhasil ditambahkan:")
print(lokasi_file_static_analyzer)

Pembersih URL File Static Analyzer berhasil ditambahkan:
C:\Users\ASUS\PHISHING\src\file_static_analyzer.py


## Membuat src/run_phishrisk.py

In [4]:
kode_run_phishrisk = r'''
from pathlib import Path
import argparse
import sys
import pandas as pd


DIREKTORI_FILE = Path(__file__).resolve()
DIREKTORI_SRC = DIREKTORI_FILE.parent
DIREKTORI_PROJECT = DIREKTORI_SRC.parent

if str(DIREKTORI_SRC) not in sys.path:
    sys.path.append(str(DIREKTORI_SRC))

import phishrisk_engine_v3


def buat_folder_output(direktori_project):
    direktori_output = Path(direktori_project) / "reports" / "outputs"
    direktori_output.mkdir(parents=True, exist_ok=True)
    return direktori_output


def simpan_dataframe(dataframe, lokasi_output):
    lokasi_output = Path(lokasi_output)
    lokasi_output.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(lokasi_output, index=False, encoding="utf-8")
    return lokasi_output


def cetak_ringkasan_url(data_hasil):
    if data_hasil.empty:
        print("Tidak ada hasil URL.")
        return

    print("\nRINGKASAN HASIL URL")
    print("=" * 60)

    ringkasan = data_hasil.groupby(
        ["hasil_akhir", "kategori_risiko"]
    ).size().reset_index(name="jumlah_data")

    print(ringkasan.to_string(index=False))

    print("\nHASIL DETAIL")
    kolom = [
        "url",
        "domain",
        "label_model",
        "skor_model",
        "skor_final",
        "kategori_risiko",
        "hasil_akhir",
        "intelligence_status",
        "rekomendasi"
    ]

    kolom_tersedia = [item for item in kolom if item in data_hasil.columns]
    print(data_hasil[kolom_tersedia].to_string(index=False))


def cetak_ringkasan_file(data_hasil_file):
    if data_hasil_file.empty:
        print("Tidak ada hasil file.")
        return

    print("\nRINGKASAN HASIL FILE")
    print("=" * 60)

    ringkasan = data_hasil_file.groupby(
        ["hasil_akhir_file_v3", "kategori_final_file_v3"]
    ).size().reset_index(name="jumlah_data")

    print(ringkasan.to_string(index=False))

    print("\nHASIL DETAIL FILE")
    kolom = [
        "nama_file",
        "ekstensi",
        "jumlah_url",
        "jumlah_url_berisiko_v3",
        "jumlah_kata_mencurigakan",
        "skor_final_file_v3",
        "kategori_final_file_v3",
        "hasil_akhir_file_v3",
        "rekomendasi_final_file_v3"
    ]

    kolom_tersedia = [item for item in kolom if item in data_hasil_file.columns]
    print(data_hasil_file[kolom_tersedia].to_string(index=False))


def baca_csv_url(lokasi_csv, nama_kolom_url=None):
    lokasi_csv = Path(lokasi_csv)

    if not lokasi_csv.exists():
        raise FileNotFoundError(f"File CSV tidak ditemukan: {lokasi_csv}")

    data = pd.read_csv(lokasi_csv)

    if nama_kolom_url and nama_kolom_url in data.columns:
        kolom_url = nama_kolom_url
    else:
        kandidat_kolom = ["url", "URL", "alamat", "Alamat", "link", "Link"]

        kolom_url = None
        for kandidat in kandidat_kolom:
            if kandidat in data.columns:
                kolom_url = kandidat
                break

        if kolom_url is None:
            raise ValueError("Kolom URL tidak ditemukan. Gunakan nama kolom: url, URL, alamat, atau link.")

    daftar_url = (
        data[kolom_url]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    daftar_url = [url for url in daftar_url if url]

    return daftar_url


def ambil_file_dari_folder(lokasi_folder):
    lokasi_folder = Path(lokasi_folder)

    if not lokasi_folder.exists():
        raise FileNotFoundError(f"Folder tidak ditemukan: {lokasi_folder}")

    daftar_file = [
        item
        for item in lokasi_folder.iterdir()
        if item.is_file()
    ]

    return daftar_file


def mode_url(engine, nilai_input, lokasi_output):
    hasil = engine.analisis_url(nilai_input)
    data_hasil = pd.DataFrame([hasil])

    cetak_ringkasan_url(data_hasil)
    simpan_dataframe(data_hasil, lokasi_output)

    print("\nOutput disimpan:")
    print(lokasi_output)


def mode_urls(engine, nilai_input, lokasi_output, nama_kolom_url=None):
    daftar_url = baca_csv_url(nilai_input, nama_kolom_url)
    data_hasil = engine.analisis_banyak_url(daftar_url)

    cetak_ringkasan_url(data_hasil)
    simpan_dataframe(data_hasil, lokasi_output)

    print("\nOutput disimpan:")
    print(lokasi_output)


def mode_file(engine, nilai_input, lokasi_output):
    hasil_file, data_url = engine.analisis_file(nilai_input)
    data_hasil_file = pd.DataFrame([hasil_file])

    cetak_ringkasan_file(data_hasil_file)
    simpan_dataframe(data_hasil_file, lokasi_output)

    if not data_url.empty:
        lokasi_url_file = lokasi_output.with_name(lokasi_output.stem + "_url_dalam_file.csv")
        simpan_dataframe(data_url, lokasi_url_file)
        print("\nOutput URL dalam file disimpan:")
        print(lokasi_url_file)

    print("\nOutput file disimpan:")
    print(lokasi_output)


def mode_folder(engine, nilai_input, lokasi_output):
    daftar_file = ambil_file_dari_folder(nilai_input)
    data_hasil_file, data_url = engine.analisis_banyak_file(daftar_file)

    cetak_ringkasan_file(data_hasil_file)
    simpan_dataframe(data_hasil_file, lokasi_output)

    if not data_url.empty:
        lokasi_url_file = lokasi_output.with_name(lokasi_output.stem + "_url_dalam_file.csv")
        simpan_dataframe(data_url, lokasi_url_file)
        print("\nOutput URL dalam file disimpan:")
        print(lokasi_url_file)

    print("\nOutput folder disimpan:")
    print(lokasi_output)


def main():
    parser = argparse.ArgumentParser(
        description="PhishRisk CLI untuk analisis URL dan file phishing secara defensif."
    )

    parser.add_argument(
        "--mode",
        required=True,
        choices=["url", "urls", "file", "folder"],
        help="Mode analisis: url, urls, file, atau folder."
    )

    parser.add_argument(
        "--input",
        required=True,
        help="Input berupa URL, file CSV, file dokumen, atau folder."
    )

    parser.add_argument(
        "--output",
        default=None,
        help="Lokasi output CSV. Jika kosong, output otomatis masuk reports/outputs."
    )

    parser.add_argument(
        "--url-column",
        default=None,
        help="Nama kolom URL jika mode urls memakai CSV."
    )

    direktori_output = buat_folder_output(DIREKTORI_PROJECT)

    if parser.parse_args().output:
        lokasi_output = Path(parser.parse_args().output)
    else:
        nama_default = {
            "url": "hasil_cli_url.csv",
            "urls": "hasil_cli_banyak_url.csv",
            "file": "hasil_cli_file.csv",
            "folder": "hasil_cli_folder.csv"
        }[parser.parse_args().mode]

        lokasi_output = direktori_output / nama_default

    args = parser.parse_args()

    engine = phishrisk_engine_v3.buat_engine(DIREKTORI_PROJECT)

    if args.mode == "url":
        mode_url(engine, args.input, lokasi_output)

    elif args.mode == "urls":
        mode_urls(engine, args.input, lokasi_output, args.url_column)

    elif args.mode == "file":
        mode_file(engine, args.input, lokasi_output)

    elif args.mode == "folder":
        mode_folder(engine, args.input, lokasi_output)


if __name__ == "__main__":
    main()
'''

lokasi_run_phishrisk = direktori_src / "run_phishrisk.py"

with open(lokasi_run_phishrisk, "w", encoding="utf-8") as file:
    file.write(kode_run_phishrisk)

print("File CLI PhishRisk berhasil dibuat:")
print(lokasi_run_phishrisk)

File CLI PhishRisk berhasil dibuat:
C:\Users\ASUS\PHISHING\src\run_phishrisk.py


## Membuat Contoh CSV Input URL

In [5]:
data_contoh_url_step10 = pd.DataFrame({
    "url": [
        "https://praktikum.gunadarma.ac.id",
        "https://baak.gunadarma.ac.id",
        "https://www.bca.co.id",
        "https://www.shopee.co.id",
        "https://www.microsoft.com",
        "http://rricrosoft.com",
        "http://rnicrosoft.com",
        "http://micros0ft-login-update.test",
        "http://bca-login-update.test",
        "http://paypal-verify-account.test",
        "http://praktikum-gunadarma-login-update.test",
        "https://xn--micrsoft-q4a.test"
    ]
})

lokasi_contoh_url_step10 = direktori_examples / "input_url_step10.csv"

data_contoh_url_step10.to_csv(
    lokasi_contoh_url_step10,
    index=False,
    encoding="utf-8"
)

print("Contoh CSV input URL berhasil dibuat:")
print(lokasi_contoh_url_step10)

data_contoh_url_step10

Contoh CSV input URL berhasil dibuat:
C:\Users\ASUS\PHISHING\examples\input_url_step10.csv


,url
0,https://praktikum.gunadarma.ac.id
1,https://baak.gunadarma.ac.id
2,https://www.bca.co.id
3,https://www.shopee.co.id
4,https://www.microsoft.com
5,http://rricrosoft.com
6,http://rnicrosoft.com
7,http://micros0ft-login-update.test
8,http://bca-login-update.test
9,http://paypal-verify-account.test


## Uji Import Program CLI

In [6]:
import run_phishrisk
importlib.reload(run_phishrisk)

print("Import run_phishrisk berhasil.")
print("Lokasi file:", direktori_src / "run_phishrisk.py")

Import run_phishrisk berhasil.
Lokasi file: C:\Users\ASUS\PHISHING\src\run_phishrisk.py


## Test CLI Mode Satu URL

In [7]:
perintah_test_url = [
    sys.executable,
    str(direktori_src / "run_phishrisk.py"),
    "--mode",
    "url",
    "--input",
    "https://praktikum.gunadarma.ac.id",
    "--output",
    str(direktori_outputs / "hasil_test_cli_satu_url.csv")
]

hasil_test_url = subprocess.run(
    perintah_test_url,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(hasil_test_url.stdout)

print("STDERR:")
print(hasil_test_url.stderr)

print("Return code:", hasil_test_url.returncode)

if hasil_test_url.returncode != 0:
    raise RuntimeError("Test CLI satu URL gagal.")

STDOUT:

RINGKASAN HASIL URL
  hasil_akhir kategori_risiko  jumlah_data
Terlihat Aman          Rendah            1

HASIL DETAIL
                              url                    domain label_model  skor_model  skor_final kategori_risiko   hasil_akhir intelligence_status                                                                                                                                            rekomendasi
https://praktikum.gunadarma.ac.id praktikum.gunadarma.ac.id  Legitimate        20.4        20.4          Rendah Terlihat Aman resmi_terlihat_aman Alamat cocok dengan daftar domain resmi dan tidak menunjukkan sinyal kuat yang mencurigakan. Tetap pastikan alamat diketik langsung dari sumber resmi.

Output disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_satu_url.csv

STDERR:

Return code: 0


## Test CLI Mode Banyak URL dari CSV

In [8]:
perintah_test_banyak_url = [
    sys.executable,
    str(direktori_src / "run_phishrisk.py"),
    "--mode",
    "urls",
    "--input",
    str(lokasi_contoh_url_step10),
    "--url-column",
    "url",
    "--output",
    str(direktori_outputs / "hasil_test_cli_banyak_url.csv")
]

hasil_test_banyak_url = subprocess.run(
    perintah_test_banyak_url,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(hasil_test_banyak_url.stdout)

print("STDERR:")
print(hasil_test_banyak_url.stderr)

print("Return code:", hasil_test_banyak_url.returncode)

if hasil_test_banyak_url.returncode != 0:
    raise RuntimeError("Test CLI banyak URL gagal.")

STDOUT:

RINGKASAN HASIL URL
  hasil_akhir kategori_risiko  jumlah_data
     Berisiko   Sangat Tinggi            7
Terlihat Aman          Rendah            5

HASIL DETAIL
                                         url                                domain label_model  skor_model  skor_final kategori_risiko   hasil_akhir         intelligence_status                                                                                                                                            rekomendasi
           https://praktikum.gunadarma.ac.id             praktikum.gunadarma.ac.id  Legitimate       20.40       20.40          Rendah Terlihat Aman         resmi_terlihat_aman Alamat cocok dengan daftar domain resmi dan tidak menunjukkan sinyal kuat yang mencurigakan. Tetap pastikan alamat diketik langsung dari sumber resmi.
                https://baak.gunadarma.ac.id                  baak.gunadarma.ac.id  Legitimate       31.60       24.00          Rendah Terlihat Aman         resmi_terlihat_

## Test CLI Mode Satu File

In [9]:
lokasi_file_sample_aman = direktori_samples_metadata / "file_samples" / "contoh_catatan_aman.txt"

if not lokasi_file_sample_aman.exists():
    raise FileNotFoundError("File sample aman belum ditemukan. Jalankan notebook 06 untuk membuat sample file.")

perintah_test_file = [
    sys.executable,
    str(direktori_src / "run_phishrisk.py"),
    "--mode",
    "file",
    "--input",
    str(lokasi_file_sample_aman),
    "--output",
    str(direktori_outputs / "hasil_test_cli_satu_file.csv")
]

hasil_test_file = subprocess.run(
    perintah_test_file,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(hasil_test_file.stdout)

print("STDERR:")
print(hasil_test_file.stderr)

print("Return code:", hasil_test_file.returncode)

if hasil_test_file.returncode != 0:
    raise RuntimeError("Test CLI satu file gagal.")

STDOUT:

RINGKASAN HASIL FILE
hasil_akhir_file_v3 kategori_final_file_v3  jumlah_data
      Terlihat Aman                 Rendah            1

HASIL DETAIL FILE
              nama_file ekstensi  jumlah_url  jumlah_url_berisiko_v3  jumlah_kata_mencurigakan  skor_final_file_v3 kategori_final_file_v3 hasil_akhir_file_v3                                                                              rekomendasi_final_file_v3
contoh_catatan_aman.txt     .txt           1                       0                         0                  12                 Rendah       Terlihat Aman File terlihat rendah risiko berdasarkan pemeriksaan statis. Tetap buka hanya jika sumbernya tepercaya.

Output URL dalam file disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_satu_file_url_dalam_file.csv

Output file disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_satu_file.csv

STDERR:

Return code: 0


## Test CLI Mode Folder

In [10]:
lokasi_folder_sample = direktori_samples_metadata / "file_samples"

if not lokasi_folder_sample.exists():
    raise FileNotFoundError("Folder sample file belum ditemukan. Jalankan notebook 06 untuk membuat sample file.")

perintah_test_folder = [
    sys.executable,
    str(direktori_src / "run_phishrisk.py"),
    "--mode",
    "folder",
    "--input",
    str(lokasi_folder_sample),
    "--output",
    str(direktori_outputs / "hasil_test_cli_folder.csv")
]

hasil_test_folder = subprocess.run(
    perintah_test_folder,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(hasil_test_folder.stdout)

print("STDERR:")
print(hasil_test_folder.stderr)

print("Return code:", hasil_test_folder.returncode)

if hasil_test_folder.returncode != 0:
    raise RuntimeError("Test CLI folder gagal.")

STDOUT:

RINGKASAN HASIL FILE
hasil_akhir_file_v3 kategori_final_file_v3  jumlah_data
           Berisiko          Sangat Tinggi            6
      Terlihat Aman                 Rendah            1

HASIL DETAIL FILE
                    nama_file ekstensi  jumlah_url  jumlah_url_berisiko_v3  jumlah_kata_mencurigakan  skor_final_file_v3 kategori_final_file_v3 hasil_akhir_file_v3                                                                              rekomendasi_final_file_v3
    contoh_aplikasi_dummy.apk     .apk           1                       1                         2                 100          Sangat Tinggi            Berisiko                     File berisiko. Jangan dibuka atau dijalankan sebelum diperiksa di lingkungan aman.
contoh_arsip_mencurigakan.zip     .zip           1                       1                         4                  89          Sangat Tinggi            Berisiko                     File berisiko. Jangan dibuka atau dijalankan sebelum diperiksa di

## Baca Output Test CLI

In [11]:
daftar_output_test_cli = [
    direktori_outputs / "hasil_test_cli_satu_url.csv",
    direktori_outputs / "hasil_test_cli_banyak_url.csv",
    direktori_outputs / "hasil_test_cli_satu_file.csv",
    direktori_outputs / "hasil_test_cli_folder.csv",
    direktori_outputs / "hasil_test_cli_satu_file_url_dalam_file.csv",
    direktori_outputs / "hasil_test_cli_folder_url_dalam_file.csv"
]

hasil_baca_output_cli = []

for lokasi_file in daftar_output_test_cli:
    hasil_baca_output_cli.append({
        "nama_file": lokasi_file.name,
        "tersedia": lokasi_file.exists(),
        "ukuran_kb": round(lokasi_file.stat().st_size / 1024, 2) if lokasi_file.exists() else 0,
        "lokasi": str(lokasi_file)
    })

data_output_test_cli = pd.DataFrame(hasil_baca_output_cli)

data_output_test_cli

,nama_file,tersedia,ukuran_kb,lokasi
0,hasil_test_cli_satu_url.csv,True,0.76,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
1,hasil_test_cli_banyak_url.csv,True,5.33,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
2,hasil_test_cli_satu_file.csv,True,1.29,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
3,hasil_test_cli_folder.csv,True,5.88,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
4,hasil_test_cli_satu_file_url_dalam_file.csv,True,0.76,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
5,hasil_test_cli_folder_url_dalam_file.csv,True,4.54,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...


## Membuat Panduan Command PowerShell

In [12]:
isi_panduan_cli = f"""
PANDUAN MENJALANKAN PHISHRISK CLI
=================================

Direktori project:
{direktori_project}

Aktifkan virtual environment:
cd C:\\Users\\ASUS\\PHISHING
.\\venv\\Scripts\\Activate.ps1

1. Cek satu URL:
python src\\run_phishrisk.py --mode url --input "https://praktikum.gunadarma.ac.id"

2. Cek banyak URL dari CSV:
python src\\run_phishrisk.py --mode urls --input "examples\\input_url_step10.csv" --url-column url

3. Cek satu file:
python src\\run_phishrisk.py --mode file --input "data\\samples_metadata\\file_samples\\contoh_catatan_aman.txt"

4. Cek semua file dalam folder:
python src\\run_phishrisk.py --mode folder --input "data\\samples_metadata\\file_samples"

5. Menentukan output sendiri:
python src\\run_phishrisk.py --mode url --input "https://www.bca.co.id" --output "reports\\outputs\\hasil_bca.csv"

Catatan keamanan:
- Program hanya melakukan analisis statis.
- Program tidak menjalankan file APK, EXE, script, macro, atau file mencurigakan.
- File berisiko sebaiknya tidak dibuka langsung di perangkat utama.
"""

lokasi_panduan_cli = direktori_outputs / "panduan_menjalankan_phishrisk_cli.txt"

with open(lokasi_panduan_cli, "w", encoding="utf-8") as file:
    file.write(isi_panduan_cli)

print("Panduan CLI berhasil dibuat:")
print(lokasi_panduan_cli)

print(isi_panduan_cli)

Panduan CLI berhasil dibuat:
C:\Users\ASUS\PHISHING\reports\outputs\panduan_menjalankan_phishrisk_cli.txt

PANDUAN MENJALANKAN PHISHRISK CLI

Direktori project:
C:\Users\ASUS\PHISHING

Aktifkan virtual environment:
cd C:\Users\ASUS\PHISHING
.\venv\Scripts\Activate.ps1

1. Cek satu URL:
python src\run_phishrisk.py --mode url --input "https://praktikum.gunadarma.ac.id"

2. Cek banyak URL dari CSV:
python src\run_phishrisk.py --mode urls --input "examples\input_url_step10.csv" --url-column url

3. Cek satu file:
python src\run_phishrisk.py --mode file --input "data\samples_metadata\file_samples\contoh_catatan_aman.txt"

4. Cek semua file dalam folder:
python src\run_phishrisk.py --mode folder --input "data\samples_metadata\file_samples"

5. Menentukan output sendiri:
python src\run_phishrisk.py --mode url --input "https://www.bca.co.id" --output "reports\outputs\hasil_bca.csv"

Catatan keamanan:
- Program hanya melakukan analisis statis.
- Program tidak menjalankan file APK, EXE, script, 

## Metadata

In [13]:
metadata_step10 = {
    "nama_program": "PhishRisk CLI Utility",
    "step": "STEP 10",
    "status": "final program utility selesai",
    "direktori_project": str(direktori_project),
    "file_cli": str(direktori_src / "run_phishrisk.py"),
    "file_engine": str(direktori_src / "phishrisk_engine_v3.py"),
    "model_digunakan": str(direktori_models / "model_terbaik_intelligence_v2.pkl"),
    "fitur_digunakan": str(direktori_outputs / "daftar_fitur_intelligence_v2.json"),
    "contoh_input_csv": str(lokasi_contoh_url_step10),
    "panduan_cli": str(lokasi_panduan_cli),
    "output_test": {
        "satu_url": str(direktori_outputs / "hasil_test_cli_satu_url.csv"),
        "banyak_url": str(direktori_outputs / "hasil_test_cli_banyak_url.csv"),
        "satu_file": str(direktori_outputs / "hasil_test_cli_satu_file.csv"),
        "folder": str(direktori_outputs / "hasil_test_cli_folder.csv")
    },
    "fungsi_utama": [
        "cek_satu_url",
        "cek_banyak_url_csv",
        "cek_satu_file",
        "cek_folder_file",
        "simpan_output_csv"
    ]
}

lokasi_metadata_step10 = direktori_outputs / "metadata_step10_cli_utility.json"

with open(lokasi_metadata_step10, "w", encoding="utf-8") as file:
    json.dump(metadata_step10, file, indent=4, ensure_ascii=False)

print("Metadata STEP 10 disimpan:")
print(lokasi_metadata_step10)

metadata_step10

Metadata STEP 10 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\metadata_step10_cli_utility.json


{'nama_program': 'PhishRisk CLI Utility',
 'step': 'STEP 10',
 'status': 'final program utility selesai',
 'direktori_project': 'C:\\Users\\ASUS\\PHISHING',
 'file_cli': 'C:\\Users\\ASUS\\PHISHING\\src\\run_phishrisk.py',
 'file_engine': 'C:\\Users\\ASUS\\PHISHING\\src\\phishrisk_engine_v3.py',
 'model_digunakan': 'C:\\Users\\ASUS\\PHISHING\\models\\model_terbaik_intelligence_v2.pkl',
 'fitur_digunakan': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\daftar_fitur_intelligence_v2.json',
 'contoh_input_csv': 'C:\\Users\\ASUS\\PHISHING\\examples\\input_url_step10.csv',
 'panduan_cli': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\panduan_menjalankan_phishrisk_cli.txt',
 'output_test': {'satu_url': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\hasil_test_cli_satu_url.csv',
  'banyak_url': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\hasil_test_cli_banyak_url.csv',
  'satu_file': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\hasil_test_cli_satu_file.csv',
  'folder': 'C:\\Users\\ASUS\\PHISH

## Validasi Final

In [14]:
daftar_file_validasi_step10 = [
    direktori_src / "run_phishrisk.py",
    direktori_src / "phishrisk_engine_v3.py",
    lokasi_contoh_url_step10,
    lokasi_panduan_cli,
    lokasi_metadata_step10,
    direktori_outputs / "hasil_test_cli_satu_url.csv",
    direktori_outputs / "hasil_test_cli_banyak_url.csv",
    direktori_outputs / "hasil_test_cli_satu_file.csv",
    direktori_outputs / "hasil_test_cli_folder.csv"
]

hasil_validasi_step10 = []

for lokasi_file in daftar_file_validasi_step10:
    hasil_validasi_step10.append({
        "nama_file": lokasi_file.name,
        "lokasi": str(lokasi_file),
        "tersedia": lokasi_file.exists(),
        "ukuran_kb": round(lokasi_file.stat().st_size / 1024, 2) if lokasi_file.exists() else 0
    })

data_validasi_step10 = pd.DataFrame(hasil_validasi_step10)

lokasi_validasi_step10 = direktori_outputs / "validasi_step10_cli_utility.csv"

data_validasi_step10.to_csv(
    lokasi_validasi_step10,
    index=False,
    encoding="utf-8"
)

print("Validasi STEP 10 disimpan:")
print(lokasi_validasi_step10)

data_validasi_step10

Validasi STEP 10 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step10_cli_utility.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,run_phishrisk.py,C:\Users\ASUS\PHISHING\src\run_phishrisk.py,True,7.17
1,phishrisk_engine_v3.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py,True,17.66
2,input_url_step10.csv,C:\Users\ASUS\PHISHING\examples\input_url_step...,True,0.36
3,panduan_menjalankan_phishrisk_cli.txt,C:\Users\ASUS\PHISHING\reports\outputs\panduan...,True,1.03
4,metadata_step10_cli_utility.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,1.28
5,hasil_test_cli_satu_url.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,0.76
6,hasil_test_cli_banyak_url.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,5.33
7,hasil_test_cli_satu_file.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,1.29
8,hasil_test_cli_folder.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,5.88


## Ringkasan

In [15]:
print("RINGKASAN STEP INI")
print("=" * 60)

print("CLI utility:", direktori_src / "run_phishrisk.py")
print("Engine final:", direktori_src / "phishrisk_engine_v3.py")
print("Contoh input CSV:", lokasi_contoh_url_step10)
print("Panduan CLI:", lokasi_panduan_cli)
print("Metadata:", lokasi_metadata_step10)
print("Validasi:", lokasi_validasi_step10)

print("\nOutput test CLI:")
display(data_output_test_cli)

print("\nValidasi final:")
display(data_validasi_step10)

print("\nContoh hasil banyak URL:")
display(pd.read_csv(direktori_outputs / "hasil_test_cli_banyak_url.csv").head(20))

print("\nContoh hasil folder file:")
display(pd.read_csv(direktori_outputs / "hasil_test_cli_folder.csv").head(20))

RINGKASAN STEP INI
CLI utility: C:\Users\ASUS\PHISHING\src\run_phishrisk.py
Engine final: C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py
Contoh input CSV: C:\Users\ASUS\PHISHING\examples\input_url_step10.csv
Panduan CLI: C:\Users\ASUS\PHISHING\reports\outputs\panduan_menjalankan_phishrisk_cli.txt
Metadata: C:\Users\ASUS\PHISHING\reports\outputs\metadata_step10_cli_utility.json
Validasi: C:\Users\ASUS\PHISHING\reports\outputs\validasi_step10_cli_utility.csv

Output test CLI:


,nama_file,tersedia,ukuran_kb,lokasi
0,hasil_test_cli_satu_url.csv,True,0.76,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
1,hasil_test_cli_banyak_url.csv,True,5.33,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
2,hasil_test_cli_satu_file.csv,True,1.29,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
3,hasil_test_cli_folder.csv,True,5.88,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
4,hasil_test_cli_satu_file_url_dalam_file.csv,True,0.76,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...
5,hasil_test_cli_folder_url_dalam_file.csv,True,4.54,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...



Validasi final:


,nama_file,lokasi,tersedia,ukuran_kb
0,run_phishrisk.py,C:\Users\ASUS\PHISHING\src\run_phishrisk.py,True,7.17
1,phishrisk_engine_v3.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py,True,17.66
2,input_url_step10.csv,C:\Users\ASUS\PHISHING\examples\input_url_step...,True,0.36
3,panduan_menjalankan_phishrisk_cli.txt,C:\Users\ASUS\PHISHING\reports\outputs\panduan...,True,1.03
4,metadata_step10_cli_utility.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,1.28
5,hasil_test_cli_satu_url.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,0.76
6,hasil_test_cli_banyak_url.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,5.33
7,hasil_test_cli_satu_file.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,1.29
8,hasil_test_cli_folder.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,5.88



Contoh hasil banyak URL:


,url,domain,tld,probabilitas_model,skor_model,label_model,skor_final,kategori_risiko,hasil_akhir,rekomendasi,intelligence_status,intelligence_reason,is_official_domain,official_brand,official_domain,brand_detected,brand_but_not_official,suspicious_keywords,suspicious_keyword_score,lookalike_brand_detected,lookalike_brand,lookalike_score,uses_punycode,uses_digit_substitution,hyphen_count
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,id,0.204000,20.40,Legitimate,20.40,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.,1,Gunadarma,baak.gunadarma.ac.id,Gunadarma,0,NaN,0,0,NaN,0.0000,0,0,0
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,id,0.316000,31.60,Legitimate,24.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.,1,Gunadarma,baak.gunadarma.ac.id,Gunadarma,0,NaN,0,0,NaN,0.0000,0,0,0
2,https://www.bca.co.id,www.bca.co.id,id,0.040543,4.05,Legitimate,4.05,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.,1,BCA,bca.co.id,BCA,0,NaN,0,0,NaN,0.0000,0,0,0
3,https://www.shopee.co.id,www.shopee.co.id,id,0.092183,9.22,Legitimate,9.22,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.,1,Shopee,shopee.co.id,Shopee,0,NaN,0,0,NaN,0.0000,0,0,0
4,https://www.microsoft.com,www.microsoft.com,com,0.480013,48.00,Legitimate,24.00,Rendah,Terlihat Aman,Alamat cocok dengan daftar domain resmi dan ti...,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.,1,Microsoft,microsoft.com,Microsoft,0,NaN,0,0,NaN,0.0000,0,0,0
5,http://rricrosoft.com,rricrosoft.com,com,0.996000,99.60,Phishing,99.60,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,domain_mirip_brand,Domain terlihat mirip dengan brand resmi.,0,NaN,NaN,NaN,0,NaN,0,1,Microsoft,0.8421,0,0,0
6,http://rnicrosoft.com,rnicrosoft.com,com,0.996000,99.60,Phishing,99.60,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,domain_mirip_brand,Domain terlihat mirip dengan brand resmi.,0,NaN,NaN,NaN,0,NaN,0,1,Microsoft,0.8421,0,0,0
7,http://micros0ft-login-update.test,micros0ft-login-update.test,test,0.988000,98.80,Phishing,98.80,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,tiruan_brand_berisiko,"URL mengandung nama brand, tetapi tidak berada...",0,NaN,NaN,Microsoft,1,"login, update",6,1,Microsoft,1.0000,0,1,2
8,http://bca-login-update.test,bca-login-update.test,test,0.996000,99.60,Phishing,99.60,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,tiruan_brand_berisiko,"URL mengandung nama brand, tetapi tidak berada...",0,NaN,NaN,BCA,1,"login, update",6,1,BCA,1.0000,0,0,2
9,http://paypal-verify-account.test,paypal-verify-account.test,test,1.000000,100.00,Phishing,100.00,Sangat Tinggi,Berisiko,Alamat terindikasi meniru brand atau domain re...,tiruan_brand_berisiko,"URL mengandung nama brand, tetapi tidak berada...",0,NaN,NaN,PayPal,1,"account, verify",5,1,PayPal,1.0000,0,0,2



Contoh hasil folder file:


,nama_file,lokasi_file,ekstensi,ukuran_kb,sha256,magic_file,magic_mismatch,risiko_awal_file,keterangan_risiko_awal,ekstensi_ganda,jumlah_url,url_terdeteksi,jumlah_url_berisiko_intelligence,jumlah_kata_mencurigakan,kata_mencurigakan,skor_kata_mencurigakan,jumlah_file_dalam_arsip,jumlah_file_berbahaya_dalam_arsip,file_berbahaya_dalam_arsip,memiliki_macro_indicator,memiliki_embedded_object,pdf_javascript,pdf_open_action,pdf_launch_action,pdf_embedded_file,jumlah_permission_apk,jumlah_izin_apk_berisiko,izin_apk_berisiko,skor_risiko_file,kategori_risiko_file,alasan_file,rekomendasi_file,jumlah_url_berisiko_v3,jumlah_url_perlu_tinjauan_v3,skor_final_file_v3,kategori_final_file_v3,hasil_akhir_file_v3,rekomendasi_final_file_v3
0,contoh_aplikasi_dummy.apk,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.apk,0.45,432bd7995a047f4d8cfcc4e1b22a5bf53ebbcfd8b4da68...,zip,0,tinggi,APK perlu diperiksa karena dapat meminta izin ...,0,1,http://ovo-login-update.test,1,2,"login, update",12,1,0,NaN,0,0,0,0,0,0,3,3,"android.permission.READ_SMS, android.permissio...",100,Sangat Tinggi,File mengandung URL. Ada URL di dalam file yan...,File sangat berisiko. Jangan dibuka atau dijal...,1,0,100,Sangat Tinggi,Berisiko,File berisiko. Jangan dibuka atau dijalankan s...
1,contoh_arsip_mencurigakan.zip,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.zip,0.30,c64326a49f0063d3a06ba762331c4b78057432214e692a...,zip,0,sedang,Arsip perlu diperiksa daftar isinya tanpa diek...,0,1,http://paypal-verify-account.test,1,4,"account, invoice, login, verify",18,2,1,invoice.pdf.exe,0,0,0,0,0,0,0,0,NaN,89,Sangat Tinggi,File mengandung URL. Ada URL di dalam file yan...,File sangat berisiko. Jangan dibuka atau dijal...,1,0,89,Sangat Tinggi,Berisiko,File berisiko. Jangan dibuka atau dijalankan s...
2,contoh_catatan_aman.txt,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.txt,0.08,39d326a4f4dde93113c861095ad982f8f95016eac868fe...,tidak_diketahui,0,rendah,TXT aman dibaca tetapi tetap perlu dicek jika ...,0,1,https://praktikum.gunadarma.ac.id,0,0,NaN,0,0,0,NaN,0,0,0,0,0,0,0,0,NaN,12,Rendah,File mengandung URL.,File terlihat rendah risiko berdasarkan pemeri...,0,0,12,Rendah,Terlihat Aman,File terlihat rendah risiko berdasarkan pemeri...
3,contoh_dokumen_link.docx,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.docx,0.50,9386f11ef2a6ec4de5c5f1467c6c7e67fe4e100a8b2f02...,zip,0,sedang,File Word modern perlu diperiksa jika mengandu...,0,2,http://shopee-login-update.test | http://bca-l...,2,4,"account, login, update, verify",22,2,0,NaN,0,0,0,0,0,0,0,0,NaN,93,Sangat Tinggi,File mengandung URL. Ada URL di dalam file yan...,File sangat berisiko. Jangan dibuka atau dijal...,2,0,93,Sangat Tinggi,Berisiko,File berisiko. Jangan dibuka atau dijalankan s...
4,contoh_halaman_login.html,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.html,0.25,465dc10139673aad627b5830ebf815b6c615e33b53aaa9...,tidak_diketahui,0,sedang,HTML perlu diperiksa jika mengandung form logi...,0,1,http://paypal-verify-account.test/login,1,5,"account, login, otp, password, verify",16,0,0,NaN,0,0,0,0,0,0,0,0,NaN,74,Tinggi,File mengandung URL. Ada URL di dalam file yan...,"Jangan langsung membuka file. Periksa sumber, ...",1,0,80,Sangat Tinggi,Berisiko,File berisiko. Jangan dibuka atau dijalankan s...
5,contoh_pdf_link.pdf,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.pdf,0.11,f5232caf9b7a45d0d8a93dab4b46e5dc4d34d0fd982374...,pdf,0,sedang,"PDF perlu diperiksa jika mengandung link, Java...",0,1,https://micros0ft-login-update.test,1,2,"login, update",12,0,0,NaN,0,0,1,1,0,0,0,0,NaN,100,Sangat Tinggi,File mengandung URL. Ada URL di dalam file yan...,File sangat berisiko. Jangan dibuka atau dijal...,1,0,100,Sangat Tinggi,Berisiko,File berisiko. Jangan dibuka atau dijalankan s...
6,contoh_pesan_mencurigakan.txt,C:\Users\ASUS\PHISHING\data\samples_metadata\f...,.txt,0.09,a5553e868d7ef5a9c5b59e9c3e18e4c199f989ff4b5609...,tidak_diketahui,0,rendah,TXT aman dibaca tetapi tetap perlu dicek jika ...,0,1,http://bca-login-up